# Data on the Line · Kurulum testi (5 dakika)

*Turning Demand Insights into Balanced Production · Vaka çalışması*

Bu notebook, oturumda kullanacağınız çalışma ortamının hazır olup olmadığını kontrol eder. Vakada da aynı adımları izleyeceksiniz.

1. Notebook'u size verilen bağlantıyla ya da QR kodla Colab'de açın. Veri dosyaları GitHub'dan otomatik olarak indirilir.
2. *Çalışma zamanı → Tümünü çalıştır* ile bütün hücreleri sırayla çalıştırın. Sonunda **“Her şey hazır.”** yazısını görmelisiniz.

Kendi bilgisayarınızda Jupyter kullanmak isterseniz `Kurulum_Testi` klasörünü indirip notebook'u orada açabilirsiniz.

Bir adım hata verirse çıkan mesajı oturumdan önce eğitmeninize gönderin.

In [ ]:
# Colab'de veri dosyaları GitHub'dan otomatik iner; kendi bilgisayarınızda bu adım atlanır
import importlib
import os
import shutil
import subprocess
import sys
import zipfile
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path

REPO_URL = "https://github.com/burcu-koca-kd/data-on-the-line.git"
REPO = Path("/content/data-on-the-line")
TEST_DIR = REPO / "tr/1_Hazırlık/Kurulum_Testi"


def _clone():
    try:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO)],
                       check=True, capture_output=True)
    except subprocess.CalledProcessError:
        print("✗ Dosyalar GitHub'dan indirilemedi. İnternet bağlantınızı kontrol "
              "edin, ardından notebook bağlantısını yeniden açın ya da bu hücreyi "
              "tekrar çalıştırın.")
        raise RuntimeError("Dosyalar indirilemedi (yukarıdaki mesaja bakın).") from None


def _update():
    """İndirilmiş kopyayı GitHub'daki son sürüme getirir; sizin dosyalarınıza dokunmaz."""
    git = ["git", "-C", str(REPO)]
    fetched = subprocess.run(git + ["fetch", "-q", "--depth", "1"], capture_output=True)
    if fetched.returncode == 0:
        subprocess.run(git + ["reset", "-q", "--hard", "origin/main"], capture_output=True)


if "google.colab" in sys.modules:
    # daha önce indirilmiş kopya en son sürüme getirilir; eksikse depo baştan indirilir
    if REPO.exists():
        _update()
    if not TEST_DIR.exists():
        shutil.rmtree(REPO, ignore_errors=True)
        _clone()
    os.chdir(TEST_DIR)
print("Çalışma klasörü:", Path.cwd())
if Path("data").exists():
    print("data klasörü bulundu ✓")
else:
    print("data klasörü BULUNAMADI: klasör yolunu kontrol edin.")

In [ ]:
def _install(*packages):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f"pip install {' '.join(packages)} başarısız oldu. Hata mesajının sonu:")
        print(result.stderr[-1500:])


def _pulp_major():
    try:
        return int(version("pulp").split(".")[0])
    except PackageNotFoundError:
        return 0


def _forget_pulp():
    """PuLP'u bellekten siler; bir sonraki import yeni kurulan paketleri görür."""
    for name in [m for m in sys.modules if m == "pulp" or m.startswith("pulp.")]:
        del sys.modules[name]
    importlib.invalidate_caches()


# Colab'de PuLP ya kurulu değildir ya da 4. sürümü kuruludur; vaka 3. sürümle yazıldı
if not 2 <= _pulp_major() < 4:
    _install("pulp>=2.7,<4")
    _forget_pulp()
for module, package in (("statsmodels", "statsmodels"), ("sklearn", "scikit-learn")):
    if find_spec(module) is None:
        _install(package)

import numpy as np
import pandas as pd
import pulp
import sklearn
import statsmodels

PULP_OK = 2 <= _pulp_major() < 4
if not PULP_OK:
    print("⚠ PuLP 3 kurulamadı, hâlâ 4. sürüm yüklü. Çalışma zamanı → Oturumu "
          "yeniden başlat'ı seçip bu notebook'u baştan çalıştırın.")
print("pulp", pulp.__version__, "✓" if PULP_OK else "✗", "| pandas", pd.__version__,
      "✓ | statsmodels", statsmodels.__version__, "✓ | scikit-learn", sklearn.__version__, "✓")

In [ ]:
result = {"PuLP sürümü uygun (3. sürüm ya da daha eski)": PULP_OK}
result["Veri dosyası okunuyor"] = len(pd.read_csv("data/sample.csv")) == 3

# 45. ve 95. dakikalarda tam olarak bunu yapacaksınız
with zipfile.ZipFile("release_test.zip") as z:
    z.extractall(".", pwd=b"test")
result["Şifreli dosya açılıyor"] = Path("release/test.csv").exists()


def solves(solver):
    """Verilen çözücü küçük bir modeli çözebiliyor mu? Çözebiliyorsa True döndürür."""
    try:
        m = pulp.LpProblem("test", pulp.LpMinimize)
        x = pulp.LpVariable.dicts("x", range(3), cat="Binary")
        m += pulp.lpSum(x.values())
        m += x[0] + x[1] >= 1
        m += x[1] + x[2] >= 1
        m.solve(solver)
        return pulp.LpStatus[m.status] == "Optimal" and pulp.value(m.objective) == 1
    except Exception:
        return False


solver = "CBC" if solves(pulp.PULP_CBC_CMD(msg=False)) else None
# yedek çözücü: yalnızca CBC açılmazsa (örneğin bazı Mac'lerde) kurulur
if solver is None and find_spec("highspy") is None:
    _install("highspy")
    _forget_pulp()
    import pulp
if solver is None and hasattr(pulp, "HiGHS") and solves(pulp.HiGHS(msg=False)):
    solver = "HiGHS"
label = "Küçük bir tamsayılı model çözülüyor" + (f" ({solver})" if solver else "")
result[label] = solver is not None

for name, ok in result.items():
    print(("✓ " if ok else "✗ ") + name)
if not solver:
    print()
    print("Çözücü başlatılamadı. En kolay çözüm Colab'i kullanmak. Kendi Mac'inizde"
          " çalışacaksanız Terminal'de 'pip install highspy' komutunu çalıştırın, "
          "ardından çekirdeği yeniden başlatıp bu notebook'u baştan çalıştırın.")
print()
if all(result.values()):
    print("Her şey hazır.")
else:
    print("Bir adım başarısız oldu. Lütfen bu mesajı eğitmeninize gönderin.")